# Colab 41 - what the encoder was trained on, and why Length inverts on AA

Two questions, both raised by the Spearman heatmap.

**A. Coverage.** `build_train_pairs` makes altered copies only, with `k = round((1-t)L)`, `t ~ U(0,1)`.
Table B.4's `n_indep = 0` row says that process puts **2 pairs of 20,000 below normLev 0.30**. AA's
evaluation set is **98.7 % below 0.30**. If that holds, SNNEED is evaluated almost entirely outside
the range it was trained on, and that is a finding about the protocol, not about AA.

**B. Why Length gives -0.73 on AA.** Equation 3.2 makes the length ratio an *upper bound* on the
target, so the relation should be positive. It is positive on SS (+0.65) and 3Di (+0.48) and strongly
negative on AA - although all three collections have the **same length distribution**, because SS and
3Di are per-residue annotations of the same CATH domains (mean 117.1, median 114, sd 40.3 for all
three). So lengths cannot explain it. The hypothesis is **selection**: the evaluation set is built by
conditioning on the target, and length ratio and content agreement are two separate causes of the
target. Conditioning on a common effect couples its causes negatively.

The test is a comparison, not a plot of one thing: length ratio against the target in the **balanced**
set and in the **unbalanced** candidate sample. Selection predicts negative in the first and positive
in the second. If both are negative the hypothesis is wrong and something else is going on.

**Nothing here is expensive.** The relevance sets and balanced pair sets come from the colab40 Drive
cache; only the 200,000-pair unbalanced sample is recomputed.

In [ ]:
import os
os.chdir('/content')
!rm -rf thesis-edit-distance-nn
!git clone https://github.com/katzemelli/thesis-edit-distance-nn.git
os.chdir('/content/thesis-edit-distance-nn')
!pip install rapidfuzz --quiet

In [ ]:
import pickle, json
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from rapidfuzz.distance import Levenshtein as RFLev

DATA_DIR = 'sampledata/cath'

# --- IDENTICAL to the run of record (colab40 cell 3) ---
AA_ALPHABET = 'ACDEFGHIKLMNPQRSTVWY'; SS_ALPHABET = 'HLS'
MIN_LEN, MAX_LEN = 50, 200
RESCUED = {'4z0mC02', '3qkaE02'}
N_TRAIN, TRAIN_SEED = 30_000, 0
STRAT_CAND, STRAT_PER_BIN, PAIR_SEED = 200_000, 400, 999
RANGE_LOW, RANGE_HIGH = 0.30, 0.70
DECILES = np.linspace(0, 1, 11)

DATASETS = ['Synth', '3Di', 'SS', 'AA']
CATH     = ['AA', '3Di', 'SS']
COLOUR = {'Synth': '#FF7F0E', '3Di': '#0072B2', 'SS': '#D62728', 'AA': '#4D4D4D'}

AA_SET, SS_SET = set(AA_ALPHABET), set(SS_ALPHABET)
is_aa = lambda s: all(c in AA_SET for c in s)
is_ss = lambda s: all(c in SS_SET for c in s)

def norm_lev(a, b):
    L = max(len(a), len(b)); return 1.0 if L == 0 else 1.0 - RFLev.distance(a, b) / L

def decile_of(nl):
    return np.clip(np.digitize(nl, DECILES) - 1, 0, 9)

try:
    from google.colab import drive
    drive.mount('/content/drive'); CACHE = '/content/drive/MyDrive/thesis_artefacts'
except Exception:
    CACHE = '/content/thesis_artefacts'
print('cache:', CACHE)

In [ ]:
# Collections, verbatim from colab40's filter.
raw = pd.concat([pd.read_csv(f'{DATA_DIR}/cath_s20_train70.csv.gz'),
                 pd.read_csv(f'{DATA_DIR}/cath_s20_test30.csv.gz')],
                ignore_index=True).drop_duplicates('domain_id')
seqs3 = pd.read_csv(f'{DATA_DIR}/cath_s20_3di.csv.gz')

def _valid(seq, isstd, d):
    return (isinstance(seq, str) and isstd(seq)
            and ((MIN_LEN <= len(seq) <= MAX_LEN) or d in RESCUED))

COLL = {
    'AA':  [s for d, s in zip(raw['domain_id'], raw['aa_seq'])              if _valid(s, is_aa, d)],
    'SS':  [s for d, s in zip(raw['domain_id'], raw['ss_seq'])              if _valid(s, is_ss, d)],
    '3Di': [s for d, s in zip(seqs3['domain_id'], seqs3['3di'].astype(str)) if _valid(s, is_aa, d)],
}
assert [len(COLL[r]) for r in ['AA', 'SS', '3Di']] == [10_501, 10_497, 10_501], 'collections drifted'

STRAT_PATH = f'{CACHE}/balanced_pairs.pkl'
assert os.path.exists(STRAT_PATH), (
    f'{STRAT_PATH} missing. It is written by colab40 and holds the balanced pair sets. '
    'Without it this notebook would have to rebuild the relevance sets - run colab40 first.')
with open(STRAT_PATH, 'rb') as fh: STRAT = pickle.load(fh)
print('balanced pair sets loaded:', {k: len(v['nl']) for k, v in STRAT.items()})

In [ ]:
# The training labels, regenerated exactly as colab40 does (build_train_pairs, seed 0).
def rand_seq(abc, rng):
    L = int(rng.integers(MIN_LEN, MAX_LEN + 1))
    return ''.join(rng.choice(list(abc), size=L))

def perturb(seq, k, abc, rng):
    s = list(seq); abc = list(abc)
    for _ in range(k):
        if len(s) == 0:         op = 'ins'
        elif len(s) >= MAX_LEN: op = rng.choice(['sub', 'del'])
        else:                   op = rng.choice(['sub', 'ins', 'del'])
        if op == 'sub':
            i = rng.integers(0, len(s)); s[i] = rng.choice([c for c in abc if c != s[i]])
        elif op == 'ins':
            i = rng.integers(0, len(s) + 1); s.insert(i, rng.choice(abc))
        else:
            i = rng.integers(0, len(s)); del s[i]
    return ''.join(s)

rng = np.random.default_rng(TRAIN_SEED)
TRAIN_NL, TRAIN_RATIO = [], []
while len(TRAIN_NL) < N_TRAIN:
    sd = rand_seq(AA_ALPHABET, rng); L = len(sd)
    t = float(rng.uniform(0, 1)); k = max(0, int(round((1 - t) * L)))
    o = perturb(sd, k, AA_ALPHABET, rng)
    if 1 <= len(o) <= MAX_LEN:
        TRAIN_NL.append(norm_lev(sd, o))
        TRAIN_RATIO.append(min(len(sd), len(o)) / max(len(sd), len(o)))
TRAIN_NL = np.array(TRAIN_NL); TRAIN_RATIO = np.array(TRAIN_RATIO)

occ = np.bincount(decile_of(TRAIN_NL), minlength=10)
print('training normLev per interval:', occ.tolist())
print(f'training pairs below {RANGE_LOW}: {int((TRAIN_NL < RANGE_LOW).sum()):,} '
      f'of {N_TRAIN:,}  ({(TRAIN_NL < RANGE_LOW).mean():.3%})')
print(f'training median {np.median(TRAIN_NL):.4f}, minimum {TRAIN_NL.min():.4f}')
print('')
print('share of each EVALUATION set below that threshold:')
for f in DATASETS:
    nl = STRAT[f]['nl']
    print(f'  {f:>5}: {(nl < RANGE_LOW).mean():7.2%}  ({int((nl < RANGE_LOW).sum()):,} of {len(nl):,})')

## A. Coverage — what the encoder saw against what it is asked

If the training distribution has no mass below 0.30 and an evaluation set is almost entirely below
0.30, then the reported number for that dataset is an extrapolation. That is a statement about the
protocol and it is measurable, so it should be measured rather than argued.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
edges = np.linspace(0, 1, 41)

ax[0].hist(TRAIN_NL, bins=edges, density=True, color='0.35', label=f'training ({N_TRAIN:,})')
for f in DATASETS:
    ax[0].hist(STRAT[f]['nl'], bins=edges, density=True, histtype='step', lw=1.8,
               color=COLOUR[f], label=f'{f} evaluation')
ax[0].axvline(RANGE_LOW, ls='--', lw=1, color='0.5')
ax[0].text(RANGE_LOW, ax[0].get_ylim()[1], ' 0.30', fontsize=8, color='0.5', va='top')
ax[0].set_xlabel('normalised Levenshtein similarity'); ax[0].set_ylabel('density')
ax[0].set_title('Training coverage against the evaluation sets')
ax[0].legend(frameon=False, fontsize=8)

share = [(STRAT[f]['nl'] < RANGE_LOW).mean() for f in DATASETS]
ax[1].bar(DATASETS, share, color=[COLOUR[f] for f in DATASETS])
ax[1].axhline((TRAIN_NL < RANGE_LOW).mean(), ls='--', lw=1.2, color='0.35')
ax[1].text(3.4, (TRAIN_NL < RANGE_LOW).mean(), ' training', fontsize=8, color='0.35',
           ha='right', va='bottom')
ax[1].set_ylabel(f'share of pairs below {RANGE_LOW}')
ax[1].set_title('How much of each evaluation set is outside the trained range')
for a in ax: a.spines[['top', 'right']].set_visible(False)
plt.tight_layout(); plt.savefig('colab41_coverage.png', dpi=200, bbox_inches='tight'); plt.show()

## B. Length ratio — balanced against unbalanced

The decisive comparison. `sim_length = min/max` is scored against the target in two populations:
the **balanced** evaluation set, and the **unbalanced** 200,000-pair candidate sample it was drawn
from. Selection predicts a sign flip between them; a property of the data predicts no flip.

In [ ]:
from scipy.stats import spearmanr

def ratio_of(seqs, i, j):
    li = np.array([len(seqs[x]) for x in i], dtype=float)
    lj = np.array([len(seqs[x]) for x in j], dtype=float)
    return np.minimum(li, lj) / np.maximum(li, lj)

rows = []
UNBAL = {}
for f in CATH:
    seqs = COLL[f]; N = len(seqs)
    r = np.random.default_rng(PAIR_SEED)
    a = r.integers(0, N, STRAT_CAND); b = r.integers(0, N, STRAT_CAND)
    keep = a != b; a, b = a[keep], b[keep]
    print(f'scoring {len(a):,} unbalanced candidate pairs for {f}...')
    nl = np.array([norm_lev(seqs[x], seqs[y]) for x, y in zip(a, b)])
    ratio = ratio_of(seqs, a, b)
    UNBAL[f] = dict(nl=nl, ratio=ratio)

    P = STRAT[f]
    br = ratio_of(seqs, P['i'], P['j']); bnl = P['nl']
    far = bnl < RANGE_LOW
    rows.append(dict(dataset=f,
                     rho_balanced=spearmanr(br, bnl).correlation,
                     rho_balanced_far=spearmanr(br[far], bnl[far]).correlation if far.sum() > 10 else np.nan,
                     rho_unbalanced=spearmanr(ratio, nl).correlation,
                     rho_unbalanced_far=spearmanr(ratio[nl < RANGE_LOW], nl[nl < RANGE_LOW]).correlation))
SEL = pd.DataFrame(rows)
print('')
print('rho_balanced reproduces the heatmap; rho_unbalanced is the same statistic before selection.')
print('A sign flip between the two columns is the selection effect.')
SEL

In [ ]:
fig, ax = plt.subplots(2, 3, figsize=(15, 8), sharex=True, sharey=True)
for k, f in enumerate(CATH):
    P = STRAT[f]; br = ratio_of(COLL[f], P['i'], P['j'])
    ax[0, k].scatter(br, P['nl'], s=3, alpha=0.25, color=COLOUR[f])
    ax[0, k].set_title(f'{f} - balanced set  ($\\rho$ = {SEL.loc[k, "rho_balanced"]:+.2f})')
    u = UNBAL[f]
    sub = np.random.default_rng(0).choice(len(u['nl']), size=min(20_000, len(u['nl'])), replace=False)
    ax[1, k].scatter(u['ratio'][sub], u['nl'][sub], s=3, alpha=0.15, color=COLOUR[f])
    ax[1, k].set_title(f'{f} - unbalanced candidates  ($\\rho$ = {SEL.loc[k, "rho_unbalanced"]:+.2f})')
    ax[1, k].set_xlabel('length ratio  min/max')
for a in ax.ravel():
    a.axhline(RANGE_LOW, ls='--', lw=0.8, color='0.6')
    a.spines[['top', 'right']].set_visible(False)
ax[0, 0].set_ylabel('normalised Levenshtein'); ax[1, 0].set_ylabel('normalised Levenshtein')
plt.tight_layout(); plt.savefig('colab41_length_selection.png', dpi=200, bbox_inches='tight'); plt.show()

In [ ]:
summary = dict(
    training=dict(n=int(N_TRAIN), median=float(np.median(TRAIN_NL)), minimum=float(TRAIN_NL.min()),
                  per_interval=np.bincount(decile_of(TRAIN_NL), minlength=10).tolist(),
                  share_below_range_low=float((TRAIN_NL < RANGE_LOW).mean())),
    eval_share_below_range_low={f: float((STRAT[f]['nl'] < RANGE_LOW).mean()) for f in DATASETS},
    length_selection=SEL.to_dict('records'),
)
with open('colab41_length_and_coverage.json', 'w') as fh:
    json.dump(summary, fh, indent=2)

print('=== what this licenses ===')
print(f"  * training holds {summary['training']['share_below_range_low']:.3%} of its pairs below "
      f"{RANGE_LOW}; AA's evaluation set holds "
      f"{summary['eval_share_below_range_low']['AA']:.1%}.")
for r in summary['length_selection']:
    flip = 'FLIPS' if r['rho_balanced'] * r['rho_unbalanced'] < 0 else 'same sign'
    print(f"  * {r['dataset']:>4}: balanced {r['rho_balanced']:+.3f} vs unbalanced "
          f"{r['rho_unbalanced']:+.3f} - {flip}")
print('')
print('If AA flips and the others do not, the -0.73 is a property of the balanced set and must be')
print('reported as one. If AA is negative in BOTH, the selection hypothesis is dead - say so.')
print('')
print('Download colab41_length_and_coverage.json + colab41_coverage.png + colab41_length_selection.png')